# 01 — Preprocessing T1 + FLAIR

Standard MS preprocessing pipeline:
1. Reorient to standard (FSL `fslreorient2std`)
2. Brain extraction (FSL `bet` or ANTs `antsBrainExtraction`)
3. Bias-field correction (ANTs `N4BiasFieldCorrection`)
4. Coregister FLAIR to T1 (FSL `flirt`, rigid-body, BBR or 6 DOF)

**Inputs:** `T1.nii.gz`, `FLAIR.nii.gz` (replace with your own paths or use the example MS subject from `../data/README.md`).

In [ ]:
from pathlib import Path

DATA = Path('/neurodesktop-storage/ms-workshop-data/sub-001')
OUT  = Path('/neurodesktop-storage/ms-workshop-out/sub-001/anat')
OUT.mkdir(parents=True, exist_ok=True)

T1    = DATA / 'anat' / 'sub-001_T1w.nii.gz'
FLAIR = DATA / 'anat' / 'sub-001_FLAIR.nii.gz'
print('T1   :', T1, T1.exists())
print('FLAIR:', FLAIR, FLAIR.exists())

## 1. Reorient to standard

In [ ]:
%%bash -s "$T1" "$FLAIR" "$OUT"
ml load fsl
T1=$1; FLAIR=$2; OUT=$3
fslreorient2std $T1    $OUT/T1_reor.nii.gz
fslreorient2std $FLAIR $OUT/FLAIR_reor.nii.gz

## 2. N4 bias correction

In [ ]:
%%bash -s "$OUT"
ml load ants
OUT=$1
N4BiasFieldCorrection -d 3 -i $OUT/T1_reor.nii.gz    -o $OUT/T1_n4.nii.gz
N4BiasFieldCorrection -d 3 -i $OUT/FLAIR_reor.nii.gz -o $OUT/FLAIR_n4.nii.gz

## 3. Brain extraction (FSL BET on T1)

In [ ]:
%%bash -s "$OUT"
ml load fsl
OUT=$1
bet $OUT/T1_n4.nii.gz $OUT/T1_brain.nii.gz -R -f 0.4 -m

## 4. Coregister FLAIR → T1 (rigid 6 DOF)

In [ ]:
%%bash -s "$OUT"
ml load fsl
OUT=$1
flirt -in $OUT/FLAIR_n4.nii.gz -ref $OUT/T1_brain.nii.gz \
      -dof 6 -cost normmi \
      -out $OUT/FLAIR_in_T1.nii.gz \
      -omat $OUT/FLAIR_to_T1.mat

## Quick QC visualisation

In [ ]:
from nilearn import plotting
plotting.plot_anat(str(OUT/'T1_brain.nii.gz'), title='T1 brain')
plotting.plot_anat(str(OUT/'FLAIR_in_T1.nii.gz'), title='FLAIR registered to T1')

## Discussion

- Why coregister FLAIR to T1 rather than the other way round?
- When would you choose ANTs `antsBrainExtraction` over FSL `bet` for MS data?
- How would the pipeline change for a longitudinal study with multiple time points?